In [15]:
# --------------------- Import Libraries ---------------------
import os
import yaml
from pathlib import Path

import scanpy as sc
import anndata as ad
import squidpy as sq
import seaborn as sns
import pandas as pd
import numpy as np
import networkx as nx
import scipy.sparse as sp
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T

from vqniche.utils.initialize import initialize_data_and_model
from vqniche.dataloaders.in_memory_dataset_blob import InMemoryDatasetBlob
from vqniche.models.vqgraph import VQGraph

## Functions

### Initialize

In [ ]:
# --------------------- Configure Backend ---------------------
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision('medium')

num_cores = int(os.environ.get("LSB_DJOB_NUMPROC", 1))
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0

print(f"Number of CPU cores: {num_cores}")
print(f"Number of GPU devices: {num_gpus}")

In [ ]:
def initialize_everything(
        config_fname: str,
        adata_fname: str,
        wandb_run_dir: str,
        model_ckpt: str
    ):
    # --------------------- Load Config ---------------------
    # Read parameters from config file
    with open(config_fname, "r") as f:
        config = yaml.safe_load(f)
    
    # --------------------- Load AnnData ---------------------
    f = config['dataset']['root_data_dir'] / 'silver' / config['dataset']['dataset_name'] / adata_fname
    adata = sc.read_h5ad(f)

    # --------------------- Initialize Dataset, Model, and Trainer ---------------------
    # initialize dataset and dataloader
    data_batch, \
    datamodule_batch, \
    _, \
    _ = initialize_data_and_model(
            config
        )

    # load from from checkpoint
    ckpt_path = wandb_run_dir / 'files' / 'checkpoints' / model_ckpt
    model = VQGraph.load_from_checkpoint(
                checkpoint_path=ckpt_path,
            )
    model.eval()

    # initialize trainer
    trainer = pl.Trainer(
                    accelerator="auto",
                    devices="auto",
                    deterministic=True,
                    num_sanity_val_steps=0,
                    enable_progress_bar=False,
                )   
    return config, \
              adata, \
              data_batch, \
              datamodule_batch, \
              model, \
              trainer

### Model Training

In [ ]:
def read_train_val_metrics(
        wandb_run_dir: Path = None,
    ) -> pd.DataFrame:
    """
    This function reads all metrics logged to output log file during each training epoch and returns them as a Pandas DataFrame.
    
    Parameters
    ----------
    wandb_run_dir : str | Path
        Path to the wandb run directory containing the output log file.
        
    Returns
    -------
    df: pd.DataFrame
        DataFrame containing all metrics logged during each training epoch.
        
    Notes:
    ------
    - This expects that the log file is in the following format:
        Metrics logged in Epoch 0: 
        ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc']
        [0, 0.6931471824645996, 0.5, 0.6931471824645996, 0.5]

        Metrics logged in Epoch 1: 
        ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc']
        [1, 0.6931471824645996, 0.5, 0.6931471824645996, 0.5]
        ...
    - All other lines in the log file are ignored.
    - The metrics are various losses and accuracies logged during training and validation epochs.
    """
    logfile = wandb_run_dir / 'files' / 'output.log'
    with open(logfile, 'r') as file:
        lines = file.readlines()
    
    data = []
    columns = []
    
    for i in range(len(lines)):
        if "Metrics logged in Epoch" in lines[i]:
            # extract column names from the next line
            columns = eval(lines[i+1].strip())
            # extract values from the next next line
            values = eval(lines[i+2].strip(), {"nan": np.nan})
            data.append(values)
    
    # create DataFrame
    df = pd.DataFrame(data, columns=columns)
    
    # replace 'nan' strings with actual NaN values
    df.replace('nan', np.nan, inplace=True)

    return df

### Model Testing

In [ ]:
def build_embeddings(
        adata,
        model,
        datamodule_batch
    ):
    H_pre_vq_conv = []
    H_post_vq_conv = []
    Logits = []
    Labels = []

    for test_batch in datamodule_batch.test_dataloader():
        h_pre_vq_conv, \
        _, \
        _, \
        _, \
        codebook_embeddings, \
        _, \
        _, \
        h_post_vq_conv, \
        logits \
            = model.forward(
                    test_batch.x,
                    test_batch.edge_index
                    )
        H_pre_vq_conv.append(h_pre_vq_conv[:test_batch.batch_size])
        H_post_vq_conv.append(h_post_vq_conv[:test_batch.batch_size])
        Logits.append(logits[:test_batch.batch_size])
        Labels.append(test_batch.y[:test_batch.batch_size])

    adata.obsm['H_pre_vq_conv'] = torch.cat(H_pre_vq_conv, dim=0).detach().numpy()
    adata.obsm['H_post_vq_conv'] = torch.cat(H_post_vq_conv, dim=0).detach().numpy()
    adata.obsm['codebook_embeddings'] = codebook_embeddings.detach().numpy()
    adata.obsm['Logits'] = torch.cat(Logits, dim=0).detach().numpy()
    adata.obs['Labels'] = torch.cat(Labels, dim=0).detach().numpy()
    adata.obs['Labels'] = np.argmax(adata.obs['Labels'],axis=1).astype(str)
    
    return adata

### Plotting

In [ ]:
def plot_train_val_metrics(
        loss_df: pd.DataFrame,
        test_acc: float = None,
    ):
    """
    This function plots all metrics logged during training and validation epochs.
    
    Parameters
    ----------
    loss_df : pd.DataFrame
        DataFrame containing all metrics logged during each training epoch.
    test_acc : float
        Test accuracy of the model.
    
    Returns
    -------
    None
    """
    # Melt the dataframe to have a 'Mode' column for 'Train' and 'Val'
    df = loss_df.melt(id_vars=['epoch'], 
                        value_vars=['train_cross_entropy', 'val_cross_entropy', 
                                    'train_mse_attribute_reconstruction', 'val_mse_attribute_reconstruction', 
                                    'train_mse_adjacency_reconstruction', 'val_mse_adjacency_reconstruction', 
                                    'train_mse_commitment_loss', 'val_mse_commitment_loss', 
                                    'train_l2_codebook_loss', 'val_l2_codebook_loss', 
                                    'train_loss', 'val_loss'],
                        var_name='Loss Term', value_name='Value')

    # Create 'Mode' column
    df['Mode'] = df['Loss Term'].apply(lambda x: 'Train' if 'train' in x else 'Val')

    # Simplify 'Loss Term' column
    df['Loss Term'] = df['Loss Term'].apply(lambda x: x.replace('train_', '').replace('val_', ''))

    # Replace metric values with proper names
    metric_names = {
        'cross_entropy': 'Cross Entropy Loss',
        'mse_attribute_reconstruction': 'MSE Attr. Reconstr.',
        'mse_adjacency_reconstruction': 'MSE Adj. Reconstr.',
        'mse_commitment_loss': 'MSE Commit Loss',
        'l2_codebook_loss': 'L2 Codebook Loss',
        'loss': 'Total Loss'
    }

    df['Loss Term'] = df['Loss Term'].map(metric_names)

    # Create a column to indicate if the metric value is NaN
    df['isNaN'] = df['Value'].isna()

    # Group by 'Loss Term' and replace NaN with 1.1 * max value for each group
    df['Value'] = df.groupby('Loss Term')['Value'].transform(lambda x: x.fillna(1.1 * x.max()))
    display(df)
        
    loss_terms = df['Loss Term'].unique()

    fig, axes = plt.subplots(2, 3, figsize=(12,8))

    for ax, loss_term in zip(axes.flatten(), loss_terms):
        sns.lineplot(data=df[df['Loss Term'] == loss_term], x='epoch', y='Value', hue='Mode', style='isNaN', markers=True, dashes=False, ax=ax)

        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()

        ax.set_xlabel('Epoch')
        ax.set_ylabel(loss_term)
        ax.set_title(loss_term)
    
    fig.legend(
        handles,
        labels,
        bbox_to_anchor=(0.5,-0.08),
        loc='lower center',
        ncol=2,
    )

    fig.suptitle(f"Test Accuracy: {test_acc:.2f}", fontsize=16)

    plt.tight_layout()
    plt.show()

In [ ]:
# Scanpy processing for UMAP
def compute_umap(
        adata,
        embedding_key='h_pre_vq_conv',
        label_key='cell_type',
    ):
    sc.pp.neighbors(adata, n_neighbors=25, n_pcs=40)  # Compute neighborhood graph
    # sc.pp.pca(adata)  # Compute PCA
    # sc.pl.pca(adata, color='labels', show=False)
    sc.tl.umap(adata)  # Compute UMAP
    sc.pl.umap(adata,color='labels',show=False)
    plt.show()

## Analysis

### sss2-1b_1p

In [3]:
dataset_name = 'sss2-1b_1p'
batch_id = 1
adata_fname = 'sim1_1105fts_10000locs.h5ad'

# set path to config file
config_fname = '/lustre/scratch126/cellgen/team361/am84/VQNiche/reproducibility/config/train_model/sss2-1b_1p_vq_graphsage.yaml'

# set path to wandb run directory
wandb_run_dir = Path('/lustre/scratch126/cellgen/team361/am84/VQNiche/reproducibility/logs/sss2-1b_1p/standalone/batch=0/spatial_n_neighs_4_cell_types/trainratio=0.8/NeighborLoader_batchsize=1024_neighbors=25_10/VQGraph/wandb/offline-run-20250219_113534-qp20ccoe')

# set path to model checkpoint within wandb run directory that you want to load
model_ckpt = 'epoch=8-val_acc=0.80.ckpt'

In [ ]:
config, \
adata, \
data_batch, \
datamodule_batch, \
model, \
trainer = initialize_everything(
        config_fname,
        adata_fname,
        wandb_run_dir,
        model_ckpt
    )

In [ ]:
test_acc = trainer.test(
    model=model,
    datamodule=datamodule_batch,
)[0]['test_acc']

In [ ]:
loss_df = read_train_val_metrics(wandb_run_dir)
display(loss_df)

In [ ]:
plot_train_val_metrics(
    loss_df,
    test_acc,
    wandb_run_dir
)

In [ ]:
adata = build_embeddings(
            adata=adata,
            model=model,
            datamodule_batch=datamodule_batch
        )

In [ ]:
compute_umap(adata)

### xhs1000-39b_1p (batch 11)

In [ ]:
dataset_name = 'sss2-1b_1p'
batch_id = 1
adata_fname = f'adata_batch{batch_id}.h5ad'

config_fname = '/lustre/scratch126/cellgen/team361/am84/VQNiche/reproducibility/config/train_model/xhs1000-39b-batch11_1p_vq_graphsage.yaml'

# set path to wandb run directory
wandb_run_dir = Path('/lustre/scratch126/cellgen/team361/am84/VQNiche/reproducibility/logs/xhs1000-39b_1p/standalone/batch=3/spatial_n_neighs_4_cell_type/trainratio=0.8/NeighborLoader_batchsize=32_neighbors=25_10/VQGraph/wandb/offline-run-20250224_112418-pumvvcbl')

# set path to model checkpoint within wandb run directory that you want to load
model_ckpt = 'epoch=1-val_acc=0.96.ckpt'

In [ ]:
config, \
adata, \
data_batch, \
datamodule_batch, \
model, \
trainer = initialize_everything(
        config_fname,
        adata_fname,
        wandb_run_dir,
        model_ckpt
    )

In [ ]:
test_acc = trainer.test(
    model=model,
    datamodule=datamodule_batch,
)[0]['test_acc']

In [ ]:
loss_df = read_train_val_metrics(wandb_run_dir)
display(loss_df)

In [ ]:
plot_train_val_metrics(
    loss_df,
    test_acc,
    wandb_run_dir
)

In [ ]:
adata = build_embeddings(
            adata=adata,
            model=model,
            datamodule_batch=datamodule_batch
        )

In [ ]:
compute_umap(adata)